In [ ]:
import os
import time
import random
import json
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score, f1_score
from thop import profile
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import all routing modules (use the improved versions)
from routing_smoe import SMoELayer
from routing_micro import MICROMoELayer
from routing_expert_choice import ExpertChoiceMoELayer
from routing_adaptive import AdaptiveDynamicMoELayer
from routing_deepseek import DeepSeekMoELayer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Running on: {device}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f" Seed set to {seed}")

In [ ]:
# ==========================================
# CELL 2: CẤU HÌNH (CONFIG) & GRID SEARCH
# ==========================================
class Config:
    MODEL_NAME = "vinai/phobert-large"
    TRAIN_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\train.csv"
    VAL_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\validation.csv"
    TEST_CSV = r"D:\Study\NCKH_MoE_ Vietnamese_Natural_Language_Inference\Code\test.csv"
    MAX_LEN = 256
    NUM_LABELS = 3
    BATCH_SIZE = 12
    LR = 1.5e-5
    EPOCHS = 10
    PATIENCE = 3
    SEEDS = [42]
    NUM_EXPERTS = 8
    CAPACITY_FACTOR = 1.5
    
    # [NÂNG CẤP] Trọng số của hàm Loss cân bằng tải (Auxiliary Loss)
    ROUTING_LOSS_WEIGHT = 1.0 

    # ================= QUẢN LÝ GRID SEARCH =================
    # Khai báo các khoảng tham số muốn máy tự động thử nghiệm
    GRID_SEARCH_SPACE = {
        "expert_choice": {
            "expert_expansion": [2.0, 4.0],
            "expert_dropout": [0.1, 0.2]
        },
        "smoe": {
            "expert_expansion": [2.0, 4.0],
            "temperature": [0.5, 1.0, 2.0],
            "expert_dropout": [0.1]
        },
        "micro": {
            "expert_expansion": [2.0, 4.0],
            "expert_dropout": [0.1]
        },
        "adaptive": {
            "expert_expansion": [2.0, 4.0],
            "threshold": [0.3, 0.5, 0.7],
            "adapt_lr": [0.01, 0.05],
            "expert_dropout": [0.1]
        },
        "deepseek": {
            "num_shared_experts": [2],
            "num_routed_to_select": [2],
            "noise_level": [0.0, 0.1],
            "expert_expansion": [2.0, 4.0],
            "expert_dropout": [0.1]
        }
    }

# ================= QUẢN LÝ THƯ MỤC THỰC NGHIỆM =================
Config.BASE_DIR = "experiments/PhoBERT_large_GridSearch" 
Config.RESUME_EXPERIMENT = True  

import os
os.makedirs(Config.BASE_DIR, exist_ok=True)
existing_exps = [d for d in os.listdir(Config.BASE_DIR) if d.startswith("experiment_")]
exp_nums = [int(d.split("_")[1]) for d in existing_exps if len(d.split("_")) > 1 and d.split("_")[1].isdigit()]

if Config.RESUME_EXPERIMENT and exp_nums:
    next_exp = max(exp_nums)
    print(f"🔄 CHẾ ĐỘ RESUME: Chạy tiếp tục tại phiên thực nghiệm {next_exp}")
else:
    next_exp = max(exp_nums) + 1 if exp_nums else 1
    print(f"📁 CHẾ ĐỘ NEW: Đã tạo phiên thực nghiệm mới experiment_{next_exp}")

Config.EXP_DIR = os.path.join(Config.BASE_DIR, f"experiment_{next_exp}")
os.makedirs(Config.EXP_DIR, exist_ok=True)

Config.CHECKPOINT_DIR = Config.EXP_DIR
Config.TRAIN_LOG_CSV = os.path.join(Config.EXP_DIR, "training_log.csv")
Config.RESULTS_CSV = os.path.join(Config.EXP_DIR, "grid_search_results.csv")

In [ ]:
label_map = {'entailment': 0, 'neutral': 1, 'contradiction': 2}
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

class NLIDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.labels = torch.tensor([label_map.get(str(l).strip().lower(), 1) for l in df['label']], dtype=torch.long)
        print(f"Pre-tokenizing {len(df)} samples...")
        self.encodings = tokenizer(
            df['premise'].astype(str).tolist(),
            df['hypothesis'].astype(str).tolist(),
            max_length=max_len, padding='max_length', truncation=True, return_tensors='pt'
        )
        print(" Tokenization done!")

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

df_train = pd.read_csv(Config.TRAIN_CSV).dropna().reset_index(drop=True)
df_val = pd.read_csv(Config.VAL_CSV).dropna().reset_index(drop=True)
df_test = pd.read_csv(Config.TEST_CSV).dropna().reset_index(drop=True)

train_loader = DataLoader(NLIDataset(df_train, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(NLIDataset(df_val, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=True)
test_loader = DataLoader(NLIDataset(df_test, tokenizer, Config.MAX_LEN), batch_size=Config.BATCH_SIZE, pin_memory=True)

In [ ]:
# ==========================================
# CELL 4: UTILS & ARCHITECTURE (NÂNG CẤP)
# ==========================================
class CheckpointManager:
    def __init__(self, model, optimizer, scheduler, scaler, model_name="moe"):
        self.model = model
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = scaler
        self.model_name = model_name
        self.best_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_best.pt")
        self.last_checkpoint_path = os.path.join(Config.CHECKPOINT_DIR, f"{model_name}_last.pt")
        self.train_log_path = Config.TRAIN_LOG_CSV
        
        if not os.path.exists(self.train_log_path):
            df = pd.DataFrame(columns=["Seed", "Epoch", "Routing", "Config", "Val_Acc", "Val_F1", "Val_Runtime_ms", "Val_VRAM_MB", "Val_Entropy", "Val_Expert_Usage"])
            df.to_csv(self.train_log_path, index=False)

    def save_checkpoint(self, epoch, val_f1, val_acc, is_best=False):
        state = {
            'epoch': epoch, 
            'model_state': self.model.state_dict(),
            'optimizer_state': self.optimizer.state_dict(),
            'scheduler_state': self.scheduler.state_dict(),
            'scaler_state': self.scaler.state_dict(),
            'best_val_f1': val_f1,
            'best_val_acc': val_acc
        }
        torch.save(state, self.last_checkpoint_path) 
        if is_best: 
            torch.save(state, self.best_checkpoint_path)

    def load_checkpoint(self):
        start_epoch, best_val_f1, best_val_acc = 0, 0.0, 0.0
        if os.path.exists(self.last_checkpoint_path):
            state = torch.load(self.last_checkpoint_path, map_location=device)
            clean_state_dict = {k: v for k, v in state['model_state'].items() if 'total_ops' not in k and 'total_params' not in k}
            self.model.load_state_dict(clean_state_dict, strict=False)
            if 'optimizer_state' in state:
                self.optimizer.load_state_dict(state['optimizer_state'])
                self.scheduler.load_state_dict(state['scheduler_state'])
                self.scaler.load_state_dict(state['scaler_state'])
            start_epoch = state['epoch'] + 1
            best_val_f1 = state.get('best_val_f1', 0.0)
            best_val_acc = state.get('best_val_acc', 0.0)
            print(f"🔋 Đã khôi phục {self.model_name}! Chạy tiếp từ Epoch {start_epoch + 1}...")
        return start_epoch, best_val_f1, best_val_acc

    def log_training(self, row):
        df = pd.read_csv(self.train_log_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(self.train_log_path, index=False)

def get_backbone_info(model):
    param_count = sum(p.numel() for p in model.backbone.parameters())
    param_memory_mb = sum(p.nelement() * p.element_size() for p in model.backbone.parameters()) / (1024 * 1024)
    return param_count, param_memory_mb

def calculate_routing_metrics(model):
    metrics = {"entropy": 0.0, "expert_usage_distribution": None}
    try:
        moe = model.moe_layer
        if hasattr(moe, "gate_logits") and moe.gate_logits is not None:
            probs = torch.softmax(moe.gate_logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-9)).sum(dim=-1).mean()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = probs.mean(dim=0).cpu().numpy().tolist()
        elif hasattr(moe, "expert_usage") and moe.expert_usage is not None:
            usage = moe.expert_usage.float()
            probs = usage / (usage.sum() + 1e-9)
            entropy = -(probs * torch.log(probs + 1e-9)).sum()
            metrics["entropy"] = entropy.item()
            metrics["expert_usage_distribution"] = usage.cpu().numpy().tolist()
    except Exception: pass
    return metrics

class LayerAttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(hidden_size, hidden_size), nn.Tanh(), nn.Linear(hidden_size, 1))

    def forward(self, hidden_states, attention_mask):
        attn_weights = self.attention(hidden_states).squeeze(-1)
        min_val = torch.finfo(attn_weights.dtype).min
        attn_weights = attn_weights.masked_fill(attention_mask == 0, min_val)
        weights = F.softmax(attn_weights, dim=-1)
        return torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)

class UnifiedMoENLI(nn.Module):
    def __init__(self, config, routing_type, routing_kwargs):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(config.MODEL_NAME)
        hidden_size = self.backbone.config.hidden_size
        self.routing_type = routing_type

        # Truyền toàn bộ cấu hình từ Grid Search (routing_kwargs) thẳng vào file .py
        if routing_type == "expert_choice": 
            self.moe_layer = ExpertChoiceMoELayer(hidden_size, config.NUM_EXPERTS, config.CAPACITY_FACTOR, **routing_kwargs)
        elif routing_type == "smoe": 
            self.moe_layer = SMoELayer(hidden_size, config.NUM_EXPERTS, **routing_kwargs)
        elif routing_type == "micro": 
            self.moe_layer = MICROMoELayer(hidden_size, config.NUM_EXPERTS, **routing_kwargs)
        elif routing_type == "adaptive": 
            self.moe_layer = AdaptiveDynamicMoELayer(hidden_size, config.NUM_EXPERTS, **routing_kwargs)
        elif routing_type == "deepseek": 
            num_shared = routing_kwargs.pop('num_shared_experts', 2)
            self.moe_layer = DeepSeekMoELayer(hidden_size, num_shared_experts=num_shared, num_routed_experts=config.NUM_EXPERTS - num_shared, **routing_kwargs)
        else: raise ValueError("Invalid Routing Type")

        self.pooling = LayerAttentionPooling(hidden_size)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden_size, hidden_size // 2), nn.GELU(), nn.Linear(hidden_size // 2, config.NUM_LABELS))

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        
        # Hứng cả output và aux_loss từ 5 file .py đã được nâng cấp
        moe_output, aux_loss = self.moe_layer(sequence_output)
        
        pooled = self.pooling(moe_output, attention_mask)
        logits = self.classifier(pooled)
        return logits, aux_loss

In [ ]:
# ==========================================
# CELL 5: MEGA PIPELINE - AUTO GRID SEARCH
# ==========================================
import itertools

for seed in Config.SEEDS:
    set_seed(seed)
    
    # Duyệt qua từng phương pháp và không gian tham số của nó
    for routing_type, param_space in Config.GRID_SEARCH_SPACE.items():
        keys = param_space.keys()
        values = param_space.values()
        
        # Sinh ra tất cả các tổ hợp tham số có thể có (Grid)
        combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
        
        for combo in combinations:
            # Tạo chuỗi định danh cho bộ tham số này (VD: exp2.0_drop0.1_temp1.0)
            combo_str = "_".join([f"{k.split('_')[-1]}{v}" for k, v in combo.items()])
            config_str = json.dumps(combo) # Để lưu vào file log
            
            print(f"\n{'='*75}\n🚀 SEED {seed} | ROUTING: {routing_type.upper()} | CONFIG: {combo_str}\n{'='*75}")
            
            # --- KIỂM TRA & BỎ QUA NẾU ĐÃ CHẠY XONG ---
            is_completed = False
            if os.path.exists(Config.RESULTS_CSV):
                try:
                    df_check = pd.read_csv(Config.RESULTS_CSV)
                    if not df_check[(df_check['Seed'] == seed) & 
                                  (df_check['Routing'] == routing_type) & 
                                  (df_check['Config_Str'] == config_str)].empty:
                        is_completed = True
                except: pass
                
            if is_completed:
                print(f"⏭️ Bỏ qua vì bộ tham số này đã hoàn thành ở lần chạy trước!")
                continue

            # --- KHỞI TẠO MÔ HÌNH ---
            model = UnifiedMoENLI(Config(), routing_type, combo).to(device)
            optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=Config.LR, weight_decay=0.01)
            total_steps = len(train_loader) * Config.EPOCHS
            scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
            scaler = GradScaler()
            criterion = nn.CrossEntropyLoss()
            
            model_name = f"phobert_large_{routing_type}_{combo_str}_seed{seed}"
            checkpoint_manager = CheckpointManager(model, optimizer, scheduler, scaler, model_name=model_name)
            start_epoch, best_val_f1, best_val_acc = checkpoint_manager.load_checkpoint()

            dummy_ids = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
            dummy_mask = torch.ones(1, Config.MAX_LEN, dtype=torch.long).to(device)
            macs, _ = profile(model, inputs=(dummy_ids, dummy_mask), verbose=False)
            gflops = (macs * 2) / 1e9

            early_stop_counter = 0

            # --- VÒNG LẶP HUẤN LUYỆN ---
            for epoch in range(start_epoch, Config.EPOCHS):
                model.train()
                train_iterator = tqdm(train_loader, desc=f"Ep {epoch+1}/{Config.EPOCHS} [{routing_type}]", leave=False)
                for batch in train_iterator:
                    ids = batch['input_ids'].to(device, non_blocking=True)
                    mask = batch['attention_mask'].to(device, non_blocking=True)
                    labels = batch['labels'].to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)
                    with autocast():
                        # Hứng aux_loss và cộng trực tiếp bằng cấu hình trọng số
                        logits, aux_loss = model(ids, mask)
                        ce_loss = criterion(logits, labels)
                        loss = ce_loss + (Config.ROUTING_LOSS_WEIGHT * aux_loss)

                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    train_iterator.set_postfix(loss=f"{loss.item():.4f}")

                # --- VALIDATION ---
                model.eval()
                val_preds, val_labels = [], []
                torch.cuda.synchronize()
                start_time = time.time()
                
                with torch.inference_mode():
                    for batch in val_loader:
                        ids = batch['input_ids'].to(device, non_blocking=True)
                        mask = batch['attention_mask'].to(device, non_blocking=True)
                        labels = batch['labels'].to(device, non_blocking=True)
                        with autocast():
                            logits, _ = model(ids, mask)
                        val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                        val_labels.extend(labels.cpu().numpy())
                        
                torch.cuda.synchronize()
                runtime_ms = ((time.time() - start_time) / len(val_loader)) * 1000
                vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

                val_acc = accuracy_score(val_labels, val_preds)
                val_f1 = f1_score(val_labels, val_preds, average='macro')
                routing_stats = calculate_routing_metrics(model)

                print(f"Ep {epoch+1} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | {runtime_ms:.2f} ms/b | VRAM: {vram_mb:.0f} MB")
                
                checkpoint_manager.log_training({
                    "Seed": seed, "Epoch": epoch+1, "Routing": routing_type, "Config": config_str,
                    "Val_Acc": val_acc, "Val_F1": val_f1, "Val_Runtime_ms": runtime_ms, 
                    "Val_VRAM_MB": vram_mb, "Val_Entropy": routing_stats["entropy"], "Val_Expert_Usage": str(routing_stats["expert_usage_distribution"])
                })

                is_best = val_f1 > best_val_f1
                if is_best:
                    best_val_f1 = val_f1
                    best_val_acc = val_acc
                    early_stop_counter = 0
                    print("✨ Val F1 cải thiện, lưu Best Checkpoint.")
                else:
                    early_stop_counter += 1
                    if early_stop_counter >= Config.PATIENCE:
                        print(f"🛑 Early stopping tại Epoch {epoch+1}!")
                        break
                        
                checkpoint_manager.save_checkpoint(epoch, best_val_f1, best_val_acc, is_best)

            # --- TEST EVALUATION KHI XONG BỘ THAM SỐ ---
            print(f"\n📥 Đang Test bộ tham số tốt nhất của {combo_str}...")
            try:
                state = torch.load(checkpoint_manager.best_checkpoint_path, map_location=device)
                clean_state = {k: v for k, v in state["model_state"].items() if 'total_ops' not in k and 'total_params' not in k}
                model.load_state_dict(clean_state, strict=False)
                model.eval()
                
                test_preds, test_labels = [], []
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
                test_start_time = time.time()
                
                with torch.inference_mode():
                    for batch in test_loader:
                        ids = batch['input_ids'].to(device, non_blocking=True)
                        mask = batch['attention_mask'].to(device, non_blocking=True)
                        labels = batch['labels'].to(device, non_blocking=True)
                        with autocast():
                            logits, _ = model(ids, mask)
                        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                        test_labels.extend(labels.cpu().numpy())

                torch.cuda.synchronize()
                test_runtime_ms = ((time.time() - test_start_time) / len(test_loader)) * 1000
                test_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

                test_acc = accuracy_score(test_labels, test_preds)
                test_f1 = f1_score(test_labels, test_preds, average='macro')
                test_routing_stats = calculate_routing_metrics(model)
                param_count, param_memory_mb = get_backbone_info(model)
                
                print(f"🎯 TEST ACC = {test_acc:.4f} | TEST F1 = {test_f1:.4f}\n")

                # Ghi kết quả tổng hợp vào File
                res_df = pd.DataFrame([{
                    "Seed": seed, "Routing": routing_type, "Config_Str": config_str,
                    "Best_Val_Acc": best_val_acc, "Best_Val_F1": best_val_f1,
                    "Test_Acc": test_acc, "Test_F1": test_f1, "GFLOPS": gflops, 
                    "Test_Runtime_ms": test_runtime_ms, "Test_VRAM_MB": test_vram_mb,
                    "Test_Entropy": test_routing_stats["entropy"], 
                    "Test_Expert_Usage": str(test_routing_stats["expert_usage_distribution"]),
                    "Backbone_Params": param_count, "Backbone_Memory_MB": param_memory_mb
                }])
                
                if not os.path.exists(Config.RESULTS_CSV):
                    res_df.to_csv(Config.RESULTS_CSV, index=False)
                else:
                    res_df.to_csv(Config.RESULTS_CSV, mode='a', header=False, index=False)
            except Exception as e:
                print(f"❌ Lỗi Test: {e}")

            del model, optimizer, scheduler, checkpoint_manager
            torch.cuda.empty_cache()
            gc.collect()

print(f"✅ Hoàn tất Grid Search! File kết quả nằm tại: {Config.RESULTS_CSV}")